# 🧹 Cleaning — and the decisions it forces on you

**Time:** ~20 minutes &nbsp;•&nbsp; **Goal:** find what is wrong with this file, and — harder — decide what to do about each thing.

Cleaning has a reputation as the boring bit. It isn't. Almost every cleaning step is a **decision about the biology**, disguised as a decision about code:

- 41 rows have no plant count. Drop them, or keep them?
- 33 rows are exact duplicates. Errors, or two real watches that both saw nothing?

pandas will do either. It will not tell you which is right.

## Run this first

In [ ]:
# Run this first. The raw file, straight off the clipboard.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df.shape)

## Skill 1 — Where are the holes?

`.isna()` gives you a `True`/`False` for every cell. Summed, it gives you a count per column — and this one line should be the second thing you ever run on a new file.

```python
df.isna().sum()
```

`NaN` ("not a number") is pandas' marker for missing. It is contagious: any arithmetic touching it gives `NaN` back. That is deliberate — it stops a missing value silently becoming a zero.

In [ ]:
# DEMO - only one column has holes in it
missing = df.isna().sum()
print(missing[missing > 0])

print()
print("rows affected:", df["Plants"].isna().sum(), "of", len(df))

In [ ]:
# DEMO - who is missing, and does it look random?
gaps = df[df["Plants"].isna()]

print(gaps["Crop_Generic"].value_counts())
print()
print(gaps["Year"].value_counts())

Look at that first table before you touch anything. Two thirds of the missing plant counts are **blackberry** — and blackberry on an allotment is a bramble along a fence, not a countable number of plants. The gap is not random noise. It is a category of crop where the question did not really apply.

That changes what you should do about it, and it is the kind of thing you can only see by looking.

### 🧹 DIY challenge

1. `df["Plants"].mean()` works despite the missing values — pandas skips them. Confirm this by comparing `df["Plants"].count()` with `len(df)`.
2. Build `df["Flowers_Per_Plant"] = df["Total_Flowers"] / df["Plants"]`. How many `NaN`s does the new column have, and why exactly that many?
3. Compare `df.dropna()` with `df.dropna(subset=["Plants"])`. Do they give the same number of rows here? Would they on a messier file?

> 💡 `.count()` counts non-missing values. `len()` counts rows. The difference is your missing data.

In [ ]:
# TODO - your turn

## Skill 2 — Drop, fill, or leave alone

Three options, three different claims about the world:

| what you write | what you are claiming |
|---|---|
| `df.dropna(subset=["Plants"])` | "These watches are unusable for my question." |
| `df["Plants"].fillna(0)` | "There were no plants." — here that is simply a lie. |
| `df["Plants"].fillna(df["Plants"].median())` | "A typical plot, near enough." — but you have invented 41 numbers. |
| leave it | "I am not using this column, so it cannot hurt me." |

For this survey the honest answer is the last one. We are counting **insects per flower**, and `Total_Flowers` has no gaps at all. Deleting 41 real watches to tidy a column you were not going to use is a bad trade.

In [ ]:
# DEMO - what each choice costs you
print("rows if we drop:", len(df.dropna(subset=["Plants"])), "of", len(df))

filled = df["Plants"].fillna(df["Plants"].median())
print()
print("median fill changes the mean from",
      round(df["Plants"].mean(), 2), "to", round(filled.mean(), 2))
print("...and the standard deviation from",
      round(df["Plants"].std(), 2), "to", round(filled.std(), 2))

The standard deviation moved **down** — 19.34 to 19.12. Only slightly, because 41 invented rows cannot do much damage among 1,620. But that is the direction you get every time: 41 perfectly average plots that nobody ever visited make the survey look more consistent than it was. On a file where a third of the values are missing, the same move is not slight at all.

### 🧹 DIY challenge

1. Make a copy `clean = df.copy()` and drop the rows with missing `Plants` from the copy. Check that `df` itself is unchanged.
2. Why `.copy()`? Try `clean = df` instead and change something — what happens to `df`?
3. Write one sentence you could put in a methods section describing what you decided about `Plants` and why.

> 💡 `clean = df` does **not** make a new table. It makes a second name for the same one. `.copy()` makes a new one.

In [ ]:
# TODO - your turn

## Skill 3 — The duplicates that aren't errors

`.duplicated()` marks every row that has appeared before, exactly, in every column.

```python
df.duplicated().sum()        # how many repeats
df[df.duplicated(keep=False)]  # show every member of every repeated group
```

Run it on this file and you get 33. The temptation is enormous: `df.drop_duplicates()`, one line, table is clean, move on.

Do not. Look at them first.

In [ ]:
# DEMO - look before you delete
print("rows flagged as repeats:", df.duplicated().sum())

repeats = df[df.duplicated(keep=False)]
print("rows involved in a repeated group:", len(repeats))

cols = ["Year", "Date", "Allotment", "Plot", "Crop_Generic", "Total_Flowers"] + INSECTS
repeats[cols].head(8)

In [ ]:
# DEMO - what do the repeats have in common?
print(repeats["Allotment"].value_counts())
print()
all_zero = repeats[INSECTS].sum(axis=1) == 0
print("repeats that recorded nothing at all:", all_zero.sum(), "of", len(repeats))

There it is. Within each repeated group it is the same plot, the same day, the same crop — and **53 of the 59 rows recorded nothing at all**.

A volunteer watched four separate courgette plants one afternoon. Nothing landed on any of them. Four rows, identical in every column, because there was nothing to tell them apart.

`drop_duplicates()` would throw three of them away — and every "visits per watch" number you calculated afterwards would be too high, because you deleted the empty watches and kept the busy ones.

**The rule:** a duplicate is only an error if you can say *how* it got there.

### 🧹 DIY challenge

1. How many rows would `df.drop_duplicates()` remove? Work out the mean `Bumblebees` before and after. Which direction did it move, and does that match the argument above?
2. Suppose you found two identical rows with **different** insect counts. Duplicate, error, or two real watches? What would you check?

> 💡 `keep=False` marks every member of a group; the default `keep="first"` marks only the repeats. Use `False` for looking, the default for counting.

In [ ]:
# TODO - your turn

## Skill 4 — Text that looks like a date is not a date 🏆

`Date` arrives as **text**. Sorting it happens to work, because these dates are written year-first. Everything else fails: you cannot ask what month it is, or how far apart two watches were.

```python
df["Date"] = pd.to_datetime(df["Date"])
```

Once a column is `datetime64` it grows a `.dt` accessor — the date version of the `.str` that fixed the header in notebook 1. `.dt.month_name()`, `.dt.year`, `.dt.dayofyear`.

And that lets you **check the `Month` column somebody typed by hand**, by deriving the same thing from the date.

In [ ]:
# DEMO - one line, and the column wakes up
df["Date"] = pd.to_datetime(df["Date"])
print("dtype now:", df["Date"].dtype)

print()
print("first watch:", df["Date"].min().date())
print("last watch: ", df["Date"].max().date())
print("survey days:", df["Date"].nunique())
print("span:       ", (df["Date"].max() - df["Date"].min()).days, "days")

In [ ]:
# DEMO - trust, but verify
derived = df["Date"].dt.month_name()
mismatch = df[derived != df["Month"]]

print(len(mismatch), "rows where the Month column disagrees with the Date")
mismatch[["Date", "Month", "Allotment", "Plot", "Crop_Generic"]]

Six watches at Moulsecoomb, all on **1 September 2017**, all labelled August. Somebody filled in the month at the top of the sheet when the survey round began and did not change it when the round ran over into a new month. It is not a catastrophe. It is completely ordinary, and it is in every hand-entered dataset you will ever touch.

**The lesson generalises:** when a value can be *derived* and also *typed by hand*, derive it. A derived column cannot disagree with its source.

```python
df["Month"] = df["Date"].dt.month_name()   # overwrite the hand-typed one
```

In [ ]:
# DEMO - derive it, and the disagreement is gone by construction
df["Month"] = df["Date"].dt.month_name()
df["MonthNum"] = df["Date"].dt.month

print((df["Date"].dt.month_name() != df["Month"]).sum(), "mismatches now")
print()
print(df["Month"].value_counts())

## Alphabetical is not chronological

One more trap, and it will appear on the x-axis of your first seasonal figure:

```python
sorted(df["Month"].unique())
# ['April', 'August', 'July', 'June', 'May', 'September']
```

That is what "sorted" means for text. The fix is to tell pandas the order once, with an **ordered Categorical** — a text column that knows what comes after what. Sorting then does the right thing everywhere, including inside `groupby` and on a plot axis.

In [ ]:
# DEMO - fix 2, the one worth learning
month_order = ["April", "May", "June", "July", "August", "September"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

print(df["Month"].dtype)
print()
print("now it sorts like a season:")
print(df["Month"].value_counts().sort_index())

### 🏆 DIY finale — your own cleaning script

Starting from the raw file, write a **single cell** that produces a clean table. It should:

1. read the raw CSV and strip whitespace from the column names
2. rename `Crop_Generic` → `Crop` and `Total_Flowers` → `Flowers`
3. convert `Date`, and derive `Month` from it rather than trusting the typed column
4. make `Month` an ordered Categorical
5. add `All_Insects` — the row total across the eight insect groups
6. **keep** the rows with missing `Plants`, and **keep** the duplicates
7. print the final shape and save it with `df.to_csv("pollinators_clean.csv", index=False)`

Then write two sentences saying what you decided about the missing values and the duplicates, and why. That paragraph is worth more marks than the code.

> 💡 `index=False` matters. Without it pandas writes the row numbers as an unnamed extra column, and the next person to open the file gets a mystery column called `Unnamed: 0`. You have probably already met one.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Working with missing data, in full: <https://pandas.pydata.org/docs/user_guide/missing_data.html>
- Hadley Wickham, *Tidy Data* (2014) — the paper that named the shape all this is heading towards: <https://vita.had.co.nz/papers/tidy-data.pdf>
- **Extra notebook:** `extras/02_cleaning_deeper` — renaming and translating values, a date index you can slice by month, and writing files out.

## Recap

You can now:

- find missing values with `.isna().sum()` — and, more importantly, **look at which rows they are**
- choose between dropping, filling and leaving alone, and say what each choice claims
- use `.duplicated()` to find repeats, and **not** delete them reflexively
- convert text to real dates with `pd.to_datetime()`, and check a hand-typed column against a derived one
- fix alphabetical month ordering with an ordered `Categorical`

**Next:** `03_groupby_basics` — one line that splits the table, summarises each piece, and puts it back together.